# Reto completo: del dataset al despliegue de Machine Learning
## Ruta A – Aprendizaje supervisado (clasificación)
**Google Colab + datos propios + Streamlit + GitHub + Docker + Google Cloud Compute Engine**

**Autores (pareja):** _Nombre 1_ · _Nombre 2_

Este notebook es **el único entregable de desarrollo**: al ejecutarlo de principio a fin
(`Entorno de ejecución → Ejecutar todo`) genera todo el proyecto en `/content/ml_streamlit_project`:

| Archivo | Rol |
|---|---|
| `data/datos.csv` | Dataset (Customer Churn Iran, repo del curso) |
| `ml_core.py` | Lógica común: inferencia de variables, Pipeline, evaluación, artefacto, predicción |
| `train.py` | Entrenamiento reproducible → `model.joblib` |
| `model.joblib` | Artefacto: pipeline + metadatos (features, categorías, rangos, métricas) |
| `app.py` | Streamlit: **entrenar con cualquier CSV**, predecir un caso, predecir por lotes |
| `requirements.txt`, `Dockerfile`, `.dockerignore`, `.streamlit/config.toml` | Empaquetado |
| `README.md` | Instrucciones completas de reproducción y despliegue |

Flujo (PDF §3): dataset → exploración → `train.py` → entrenar/evaluar → `model.joblib` → `app.py` →
probar Streamlit en Colab → archivos Docker → GitHub → VM GCP → Docker → `git clone` → `docker build` →
`docker run` → firewall → IP pública → prueba cruzada.

## 4. Preparación del proyecto en Google Colab
### Paso 4.1 – Instalar dependencias

In [ ]:
!pip -q install streamlit scikit-learn pandas numpy joblib openpyxl

### Paso 4.2 – Crear la carpeta del proyecto

In [ ]:
from pathlib import Path
import os, sys

PROJECT = Path("/content/ml_streamlit_project")
PROJECT.mkdir(exist_ok=True)
(PROJECT / "data").mkdir(exist_ok=True)
(PROJECT / ".streamlit").mkdir(exist_ok=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print(PROJECT)

### Paso 4.3 – Recordatorio sobre persistencia
El almacenamiento de Colab es **temporal**: un reinicio del runtime puede borrar los archivos.
Al final del notebook se genera `ml_streamlit_project.zip` para descargarlo y subirlo a GitHub.

## Configuración del experimento
Cambiando **solo esta celda** el proyecto completo se reentrena con otro CSV de clasificación
(por ejemplo `Default_Clientes.csv` del repo, objetivo `Default`, que usa `;` como separador).

In [ ]:
DATA_URL = ("https://raw.githubusercontent.com/wilmerpineda/machine_learning_ii_course_public/"
            "main/datasets/public/Customer_Churn_Iran.csv")
TARGET = "Churn"
EXTRA_DROP = ["Customer Value"]   # además de las sugeridas automáticamente (IDs, constantes)
MODEL_NAME = "random_forest"      # "logistic_regression" o "random_forest"
USE_UPLOAD = False                # True -> seleccionar un CSV/Excel propio desde el computador
RANDOM_STATE = 42

## Módulo común `ml_core.py`
Contiene el Pipeline del PDF §7.3 (imputación + escalado / imputación + OneHot + modelo),
la inferencia automática de variables numéricas/categóricas, la evaluación del PDF §8.1 y la
construcción del artefacto (PDF §8.2 y §14). Lo usan **tanto `train.py` como `app.py`**, por lo que
las transformaciones de entrenamiento y de serving son idénticas.

In [ ]:
%%writefile /content/ml_streamlit_project/ml_core.py
"""ml_core.py - Lógica común de entrenamiento e inferencia (Ruta A - supervisada).

La usan train.py (entrenamiento reproducible) y app.py (Streamlit), de modo que
las transformaciones de entrenamiento y de serving sean SIEMPRE las mismas
(Pipeline de scikit-learn: preprocesamiento + modelo en un solo objeto).
"""
from __future__ import annotations

import io
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
MAX_CAT_UNIQUE = 10        # enteros con <= 10 valores distintos se tratan como categóricos
MAX_CLASSES = 20           # más clases que esto -> probablemente es regresión
MODELS = {
    "logistic_regression": "Regresión logística",
    "random_forest": "Random Forest",
}


# ----------------------------------------------------------------------------
# Lectura y limpieza
# ----------------------------------------------------------------------------
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza nombres: quita BOM y espacios repetidos ('Call  Failure' -> 'Call Failure')."""
    df = df.copy()
    df.columns = [" ".join(str(c).replace("\ufeff", "").split()) for c in df.columns]
    return df


def read_csv_auto(source) -> pd.DataFrame:
    """Lee un CSV desde ruta, URL o archivo subido detectando el separador (',' ';' tab '|')."""
    if hasattr(source, "read"):  # archivo subido (Streamlit / Colab)
        raw = source.read()
        if isinstance(raw, bytes):
            raw = raw.decode("utf-8-sig", errors="replace")
        return clean_columns(pd.read_csv(io.StringIO(raw), sep=None, engine="python"))
    return clean_columns(pd.read_csv(source, sep=None, engine="python", encoding="utf-8-sig"))


def to_str(X):
    """Convierte categóricas a texto para que 1 y '1' sean la misma categoría."""
    return pd.DataFrame(X).astype(str).values


# ----------------------------------------------------------------------------
# Inferencia automática de variables
# ----------------------------------------------------------------------------
def infer_feature_types(X: pd.DataFrame, max_cat_unique: int = MAX_CAT_UNIQUE):
    numeric, categorical = [], []
    for col in X.columns:
        s = X[col]
        if pd.api.types.is_bool_dtype(s) or not pd.api.types.is_numeric_dtype(s):
            categorical.append(col)
        elif pd.api.types.is_integer_dtype(s) and s.nunique(dropna=True) <= max_cat_unique:
            categorical.append(col)
        else:
            numeric.append(col)
    return numeric, categorical


def suggest_drop_columns(df: pd.DataFrame, target: str) -> list[str]:
    """Sugiere excluir identificadores, constantes y texto libre de alta cardinalidad."""
    n = len(df)
    drop = []
    for col in df.columns:
        if col == target:
            continue
        s = df[col]
        nunique = s.nunique(dropna=True)
        name = col.lower()
        is_id_name = name in {"id", "index", "instant"} or name.endswith("_id") or name.startswith("id_")
        if nunique <= 1:
            drop.append(col)
        elif is_id_name:
            drop.append(col)
        elif nunique == n and not pd.api.types.is_float_dtype(s):
            drop.append(col)
        elif s.dtype == object and nunique > 0.5 * n:
            drop.append(col)
    return drop


# ----------------------------------------------------------------------------
# Pipeline (PDF §7.3)
# ----------------------------------------------------------------------------
def build_pipeline(numeric_features, categorical_features, model_name="random_forest") -> Pipeline:
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("to_str", FunctionTransformer(to_str)),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ])
    if model_name == "logistic_regression":
        classifier = LogisticRegression(max_iter=1000)
    elif model_name == "random_forest":
        # Configuración tomada del notebook de semana 14 del curso
        classifier = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Modelo no soportado: {model_name}")
    return Pipeline([("preprocessor", preprocessor), ("classifier", classifier)])


# ----------------------------------------------------------------------------
# Entrenamiento + evaluación + artefacto (PDF §7, §8, §14)
# ----------------------------------------------------------------------------
def validate_target(y: pd.Series) -> None:
    n_classes = y.nunique()
    if n_classes < 2:
        raise ValueError("La variable objetivo debe tener al menos 2 clases.")
    if n_classes > MAX_CLASSES:
        raise ValueError(
            f"La variable objetivo tiene {n_classes} valores distintos: parece un problema de "
            "regresión. Esta aplicación implementa la ruta supervisada de CLASIFICACIÓN."
        )


def evaluate(model, X_test, y_test) -> dict:
    pred = model.predict(X_test)
    classes = list(model.classes_)
    metrics = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
        "f1_weighted": float(f1_score(y_test, pred, average="weighted")),
        "recall_macro": float(recall_score(y_test, pred, average="macro")),
    }
    proba = model.predict_proba(X_test)
    try:
        if len(classes) == 2:
            metrics["roc_auc"] = float(roc_auc_score(y_test, proba[:, 1]))
        else:
            metrics["roc_auc_ovr"] = float(roc_auc_score(y_test, proba, multi_class="ovr", labels=classes))
    except ValueError:
        pass
    return {
        "metrics": metrics,
        "confusion_matrix": confusion_matrix(y_test, pred, labels=classes).tolist(),
        "classification_report": classification_report(y_test, pred, zero_division=0),
        "labels": [str(c) for c in classes],
    }


def train(
    df: pd.DataFrame,
    target: str,
    drop_columns=None,
    model_name: str = "random_forest",
    test_size: float = 0.20,
    random_state: int = RANDOM_STATE,
    max_rows: int | None = None,
    dataset_name: str = "dataset.csv",
) -> dict:
    """Entrena el pipeline completo y devuelve el artefacto (dict) listo para joblib."""
    df = clean_columns(df)
    target = " ".join(str(target).split())
    if target not in df.columns:
        raise ValueError(f"La columna objetivo '{target}' no existe en el dataset.")

    drop_columns = [" ".join(str(c).split()) for c in (drop_columns or [])]
    df = df.dropna(subset=[target])
    if max_rows and len(df) > max_rows:
        df = df.sample(max_rows, random_state=random_state)

    y = df[target]
    if y.dtype == object or pd.api.types.is_bool_dtype(y):
        y = y.astype(str)
    validate_target(y)

    features = [c for c in df.columns if c != target and c not in drop_columns]
    if not features:
        raise ValueError("No quedan variables explicativas después de excluir columnas.")
    X = df[features].copy()
    numeric_features, categorical_features = infer_feature_types(X)
    for col in categorical_features:  # categorías como texto (mismo tratamiento que en serving)
        X[col] = X[col].where(X[col].isna(), X[col].astype(str)).astype(object)

    counts = y.value_counts()
    stratify = y if counts.min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=stratify
    )

    model = build_pipeline(numeric_features, categorical_features, model_name)
    model.fit(X_train, y_train)
    evaluation = evaluate(model, X_test, y_test)

    category_options = {
        col: sorted(X_train[col].dropna().astype(str).unique().tolist())
        for col in categorical_features
    }
    numeric_stats = {
        col: {
            "min": float(X_train[col].min()),
            "max": float(X_train[col].max()),
            "median": float(X_train[col].median()),
            "integer": bool(pd.api.types.is_integer_dtype(X_train[col])),
        }
        for col in numeric_features
    }
    # Caso de referencia para verificar paridad entre entornos (patrón semana 14)
    ref_input = X_test.iloc[[0]]
    reference = {
        "input": {k: (None if pd.isna(v) else (v.item() if hasattr(v, "item") else v))
                  for k, v in ref_input.iloc[0].to_dict().items()},
        "proba": model.predict_proba(ref_input)[0].tolist(),
        "tolerance": 1e-9,
    }

    return {
        "model": model,
        "features": features,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "task": "classification",
        "target": target,
        "classes": [c.item() if hasattr(c, "item") else c for c in model.classes_],
        "category_options": category_options,
        "numeric_stats": numeric_stats,
        "dropped_columns": drop_columns,
        "model_name": model_name,
        "metrics": evaluation["metrics"],
        "evaluation": evaluation,
        "dataset": dataset_name,
        "n_rows_train": int(len(X_train)),
        "n_rows_test": int(len(X_test)),
        "sklearn_version": sklearn.__version__,
        "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "reference": reference,
    }


# ----------------------------------------------------------------------------
# Inferencia
# ----------------------------------------------------------------------------
def prepare_input(df: pd.DataFrame, artifact: dict):
    """Alinea un DataFrame nuevo con el contrato del artefacto.

    Devuelve (X, faltantes, sobrantes). Las columnas faltantes se crean vacías
    (el imputador del pipeline las completa) y las sobrantes se ignoran.
    """
    df = clean_columns(df)
    features = artifact["features"]
    missing = [c for c in features if c not in df.columns]
    extra = [c for c in df.columns if c not in features]
    X = df.reindex(columns=features)
    for col in artifact["numeric_features"]:
        X[col] = pd.to_numeric(X[col], errors="coerce")
    for col in artifact["categorical_features"]:
        X[col] = X[col].where(X[col].isna(), X[col].astype(str)).astype(object)
    return X, missing, extra


def predict_df(artifact: dict, df: pd.DataFrame):
    X, missing, extra = prepare_input(df, artifact)
    model = artifact["model"]
    out = df.copy()
    out["prediccion"] = model.predict(X)
    proba = model.predict_proba(X)
    for i, cls in enumerate(model.classes_):
        out[f"prob_{cls}"] = np.round(proba[:, i], 4)
    return out, missing, extra


def check_parity(artifact: dict) -> float:
    """Recalcula el caso de referencia; devuelve la diferencia máxima absoluta."""
    ref = artifact["reference"]
    X = pd.DataFrame([ref["input"]], columns=artifact["features"])
    proba = artifact["model"].predict_proba(X)[0]
    return float(np.max(np.abs(proba - np.array(ref["proba"]))))

## 5. Cargar los datos
Opción B (URL pública, por defecto) u Opción A (subir archivo propio).

In [ ]:
import pandas as pd
import ml_core

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    filename = next(iter(uploaded))
    print("Archivo cargado:", filename)
    if filename.lower().endswith(".csv"):
        df = ml_core.read_csv_auto(filename)
    elif filename.lower().endswith((".xlsx", ".xls")):
        df = ml_core.clean_columns(pd.read_excel(filename))
    else:
        raise ValueError("Use un archivo CSV o Excel.")
else:
    df = ml_core.read_csv_auto(DATA_URL)

df.head()

## 6. Revisión mínima del dataset
- **¿Qué representa una fila?** Un cliente de una compañía de telecomunicaciones iraní, con su uso agregado de 12 meses.
- **Objetivo:** predecir `Churn` (1 = el cliente abandona).
- **¿Qué variables pueden entrar?** Las de uso y perfil. `Customer Value` se excluye porque se calcula a partir del uso (riesgo de fuga) y no está disponible para un cliente nuevo.
- Los nombres con doble espacio (`Call  Failure`) se normalizan con `ml_core.clean_columns`.

In [ ]:
print(df.shape)
display(df.head())
display(df.dtypes)
display(df.isna().sum().sort_values(ascending=False).head(15))
display(df.describe(include="all").T)
print("Duplicados:", df.duplicated().sum())
display(df[TARGET].value_counts(normalize=True).rename("proporción"))

## 7. Ruta A – Construcción supervisada
### Paso 7.1 – Definir objetivo y variables (inferencia automática)

In [ ]:
drop_columns = ml_core.suggest_drop_columns(df, TARGET)
drop_columns += [c for c in EXTRA_DROP if c in df.columns and c not in drop_columns]

features = [c for c in df.columns if c not in drop_columns + [TARGET]]
X = df[features].copy()
y = df[TARGET].copy()
numeric_features, categorical_features = ml_core.infer_feature_types(X)

print("Excluidas:", drop_columns)
print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

### Paso 7.2 – Separar entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split

for col in categorical_features:
    X[col] = X[col].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_test.shape)

### Paso 7.3 – Preprocesamiento + modelo (Pipeline)
Modelo base del PDF: regresión logística. Se compara con Random Forest (configuración del notebook
de semana 14 del curso: 300 árboles, `min_samples_leaf=5`, `class_weight="balanced"`).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, roc_auc_score

results = {}
for name in ml_core.MODELS:
    model = ml_core.build_pipeline(numeric_features, categorical_features, name)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = {
        "f1_weighted": f1_score(y_test, pred, average="weighted"),
        "recall_clase_1": recall_score(y_test, pred, pos_label=model.classes_[-1]),
        "roc_auc": roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]) if len(model.classes_) == 2 else None,
    }
    print(f"===== {ml_core.MODELS[name]} =====")
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))

pd.DataFrame(results).T.round(4)

## 8. Evaluación y elección del modelo
Las clases están desbalanceadas (≈16 % de churn), por eso *accuracy* no basta (PDF §8.1).
Elegimos el modelo con mejor **recall de la clase churn** y **F1 ponderado**: no detectar un cliente que
se va (falso negativo) cuesta más que contactar a uno que se habría quedado. ROC-AUC sirve de apoyo
porque no depende del umbral.

In [ ]:
best = max(results, key=lambda k: (results[k]["recall_clase_1"], results[k]["f1_weighted"]))
print("Modelo elegido:", ml_core.MODELS[best])
if best != MODEL_NAME:
    print(f"Nota: MODEL_NAME está configurado como '{MODEL_NAME}'; train.py usará el elegido -> '{best}'")
MODEL_NAME = best

### Paso 8.2 – Guardar artefacto (modelo + metadatos + `category_options`)

In [ ]:
import joblib

artifact = ml_core.train(df, TARGET, drop_columns, MODEL_NAME, random_state=RANDOM_STATE,
                         dataset_name="datos.csv")
joblib.dump(artifact, PROJECT / "model.joblib")
print({k: round(v, 4) for k, v in artifact["metrics"].items()})
print("Claves del artefacto:", list(artifact))

## 11–12. Script reproducible `train.py` y dataset dentro del proyecto

In [ ]:
df.to_csv(PROJECT / "data" / "datos.csv", index=False)
print("Dataset guardado en", PROJECT / "data" / "datos.csv", df.shape)

In [ ]:
%%writefile /content/ml_streamlit_project/train.py
"""train.py - Entrenamiento reproducible (PDF §11).

Uso:
    python train.py                                   # dataset por defecto (Churn Iran)
    python train.py --data otro.csv --target clase    # cualquier CSV de clasificación
    python train.py --drop "ID" "Otra columna" --model logistic_regression
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import joblib
import pandas as pd

import ml_core

DATA_PATH = Path("data/datos.csv")
DATA_URL = (
    "https://raw.githubusercontent.com/wilmerpineda/machine_learning_ii_course_public/"
    "main/datasets/public/Customer_Churn_Iran.csv"
)
MODEL_PATH = Path("model.joblib")
DEFAULT_TARGET = "Churn"
DEFAULT_DROP = ["Customer Value"]  # derivada del uso -> riesgo de fuga de información


def load_data(path: Path) -> pd.DataFrame:
    if path.exists():
        return ml_core.read_csv_auto(path)
    print(f"{path} no existe; descargando desde {DATA_URL}")
    return ml_core.read_csv_auto(DATA_URL)


def train_model(df: pd.DataFrame, target: str, drop: list[str], model_name: str, name: str) -> dict:
    artifact = ml_core.train(df, target=target, drop_columns=drop, model_name=model_name, dataset_name=name)
    ev = artifact["evaluation"]
    print("Variables numéricas:", artifact["numeric_features"])
    print("Variables categóricas:", artifact["categorical_features"])
    print("Matriz de confusión (filas=real, columnas=predicho):", ev["labels"])
    for row in ev["confusion_matrix"]:
        print("   ", row)
    print(ev["classification_report"])
    print(json.dumps({k: round(v, 4) for k, v in artifact["metrics"].items()}, indent=2))
    return artifact


def main() -> None:
    parser = argparse.ArgumentParser(description="Entrena el pipeline supervisado y guarda model.joblib")
    parser.add_argument("--data", type=Path, default=DATA_PATH)
    parser.add_argument("--target", default=DEFAULT_TARGET)
    parser.add_argument("--drop", nargs="*", default=None,
                        help="Columnas a excluir (por defecto: sugeridas + Customer Value si existe)")
    parser.add_argument("--model", default="random_forest", choices=list(ml_core.MODELS))
    parser.add_argument("--out", type=Path, default=MODEL_PATH)
    args = parser.parse_args()

    df = load_data(args.data)
    if args.drop is None:
        drop = ml_core.suggest_drop_columns(df, args.target)
        drop += [c for c in DEFAULT_DROP if c in df.columns and c not in drop]
    else:
        drop = args.drop
    print(f"Datos: {df.shape} | objetivo: {args.target} | excluidas: {drop} | modelo: {args.model}")

    artifact = train_model(df, args.target, drop, args.model, args.data.name)
    joblib.dump(artifact, args.out)
    assert ml_core.check_parity(joblib.load(args.out)) <= 1e-9, "Fallo de paridad tras recargar"
    print(f"Modelo guardado en {args.out}")


if __name__ == "__main__":
    main()

Reentrenar desde cero con el script (PDF §16.1): debe regenerar `model.joblib` con las mismas métricas.

In [ ]:
drop_args = " ".join(f'"{c}"' for c in drop_columns)
!cd {PROJECT} && python train.py --target "{TARGET}" --model {MODEL_NAME} --drop {drop_args}
!ls -la {PROJECT}

## 13–14. Aplicación Streamlit `app.py`
Tres pestañas:
1. **Entrenar con nuevo CSV**: botón para seleccionar cualquier CSV, elegir la columna objetivo, las columnas a excluir y el modelo. Se entrena con el **mismo pipeline** de `ml_core.py`, se muestran las métricas y el modelo queda activo (y se puede descargar).
2. **Predecir un caso**: el formulario se genera desde el artefacto (`number_input` con mediana/rango y `selectbox` con las categorías vistas en entrenamiento, PDF §14).
3. **Predecir por lotes**: se sube un CSV, se validan las columnas (las faltantes se imputan y las sobrantes se ignoran) y se descarga el resultado.

Al arrancar, la app carga `model.joblib` **sin reentrenar** (PDF §34). Entrenar desde la app es una extensión sobre el PDF.

In [ ]:
%%writefile /content/ml_streamlit_project/app.py
"""app.py - Aplicación Streamlit (PDF §13-14 + extensión de entrenamiento con CSV nuevo).

Pestañas:
  1. Entrenar con un CSV nuevo (cualquier dataset de clasificación).
  2. Predecir un caso (formulario generado desde el artefacto).
  3. Predecir por lotes (CSV) y descargar resultados.
"""
from __future__ import annotations

import io
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

import ml_core

MODEL_PATH = Path("model.joblib")

st.set_page_config(page_title="ML App - Clasificación supervisada", layout="wide")


@st.cache_resource
def load_default_artifact():
    if MODEL_PATH.exists():
        return joblib.load(MODEL_PATH)
    return None


@st.cache_data(show_spinner=False)
def read_uploaded(data: bytes) -> pd.DataFrame:
    return ml_core.read_csv_auto(io.BytesIO(data))


if "artifact" not in st.session_state:
    st.session_state["artifact"] = load_default_artifact()
artifact = st.session_state["artifact"]

# ----------------------------------------------------------------------------
# Barra lateral: modelo activo
# ----------------------------------------------------------------------------
with st.sidebar:
    st.header("Modelo activo")
    if artifact is None:
        st.warning("No hay modelo cargado. Entrene uno en la pestaña 1.")
    else:
        st.write(f"**Dataset:** {artifact.get('dataset', '-')}")
        st.write(f"**Objetivo:** `{artifact['target']}`")
        st.write(f"**Modelo:** {ml_core.MODELS.get(artifact['model_name'], artifact['model_name'])}")
        st.write(f"**Clases:** {', '.join(map(str, artifact['classes']))}")
        st.write(f"**Variables:** {len(artifact['features'])}")
        st.json({k: round(v, 4) for k, v in artifact["metrics"].items()}, expanded=False)
        st.download_button(
            "Descargar model.joblib",
            data=(lambda b: (joblib.dump(artifact, b), b.getvalue())[1])(io.BytesIO()),
            file_name="model.joblib",
        )
        if MODEL_PATH.exists() and st.button("Restaurar modelo por defecto"):
            st.session_state["artifact"] = load_default_artifact()
            st.rerun()

st.title("Aplicación de Machine Learning supervisado")
st.caption("Pipeline scikit-learn (preprocesamiento + modelo) · Streamlit · Docker · GCP")

tab_train, tab_single, tab_batch = st.tabs(
    ["1 · Entrenar con nuevo CSV", "2 · Predecir un caso", "3 · Predecir por lotes (CSV)"]
)

# ----------------------------------------------------------------------------
# 1. Entrenar
# ----------------------------------------------------------------------------
with tab_train:
    st.subheader("Entrenar un modelo con cualquier CSV")
    st.write("Seleccione un archivo CSV (separador `,` o `;` detectado automáticamente).")
    up = st.file_uploader("Explorar archivos", type=["csv"], key="train_csv")
    if up is not None:
        try:
            df = read_uploaded(up.getvalue())
        except Exception as exc:  # noqa: BLE001
            st.error(f"No se pudo leer el CSV: {exc}")
            st.stop()

        c1, c2, c3 = st.columns(3)
        c1.metric("Filas", f"{df.shape[0]:,}")
        c2.metric("Columnas", df.shape[1])
        c3.metric("Celdas nulas", int(df.isna().sum().sum()))
        st.dataframe(df.head(20))
        with st.expander("Tipos y nulos por columna"):
            st.dataframe(pd.DataFrame({
                "tipo": df.dtypes.astype(str),
                "nulos": df.isna().sum(),
                "valores_distintos": df.nunique(),
            }))

        cols = df.columns.tolist()
        target = st.selectbox("Columna objetivo (y)", cols, index=len(cols) - 1)
        suggested = ml_core.suggest_drop_columns(df, target)
        drop = st.multiselect(
            "Columnas a excluir (identificadores, fuga de información...)",
            [c for c in cols if c != target], default=suggested,
        )
        cA, cB, cC = st.columns(3)
        model_name = cA.selectbox("Modelo", list(ml_core.MODELS), format_func=ml_core.MODELS.get, index=1)
        test_size = cB.slider("Proporción de prueba", 0.1, 0.4, 0.2, 0.05)
        max_rows = cC.number_input("Máximo de filas (0 = todas)", min_value=0, value=0, step=1000)

        st.write("Distribución de la variable objetivo:")
        st.dataframe(df[target].value_counts(dropna=False).rename("conteo").to_frame().T)

        if st.button("Entrenar", type="primary"):
            with st.spinner("Entrenando..."):
                try:
                    new_art = ml_core.train(
                        df, target=target, drop_columns=drop, model_name=model_name,
                        test_size=test_size, max_rows=int(max_rows) or None, dataset_name=up.name,
                    )
                except Exception as exc:  # noqa: BLE001
                    st.error(str(exc))
                    st.stop()
            st.session_state["artifact"] = new_art
            artifact = new_art
            st.success("Modelo entrenado y activado. Ya puede usar las pestañas 2 y 3.")
            ev = new_art["evaluation"]
            st.write("**Métricas en el conjunto de prueba**")
            st.dataframe(pd.Series(new_art["metrics"]).round(4).to_frame("valor").T)
            st.write("**Matriz de confusión** (filas = real, columnas = predicho)")
            st.dataframe(pd.DataFrame(ev["confusion_matrix"], index=ev["labels"], columns=ev["labels"]))
            st.code(ev["classification_report"])
            st.info(f"Numéricas: {new_art['numeric_features']}\n\nCategóricas: {new_art['categorical_features']}")
            if st.button("Actualizar interfaz"):
                st.rerun()

if artifact is None:
    with tab_single:
        st.info("Primero entrene un modelo.")
    with tab_batch:
        st.info("Primero entrene un modelo.")
    st.stop()

# ----------------------------------------------------------------------------
# 2. Un caso (PDF §13-14)
# ----------------------------------------------------------------------------
with tab_single:
    st.subheader(f"Predecir `{artifact['target']}` para un nuevo caso")
    values = {}
    with st.form("single"):
        grid = st.columns(3)
        i = 0
        for feature in artifact["numeric_features"]:
            stats = artifact["numeric_stats"][feature]
            with grid[i % 3]:
                if stats["integer"]:
                    values[feature] = st.number_input(feature, value=int(round(stats["median"])), step=1,
                                                      help=f"rango entrenamiento: {stats['min']:g} – {stats['max']:g}")
                else:
                    values[feature] = st.number_input(feature, value=float(stats["median"]),
                                                      help=f"rango entrenamiento: {stats['min']:g} – {stats['max']:g}")
            i += 1
        for feature in artifact["categorical_features"]:
            with grid[i % 3]:
                values[feature] = st.selectbox(feature, artifact["category_options"][feature])
            i += 1
        submitted = st.form_submit_button("Ejecutar modelo", type="primary")
    if submitted:
        X_new = pd.DataFrame([values], columns=artifact["features"])
        out, _, _ = ml_core.predict_df(artifact, X_new)
        st.success(f"Clase predicha: **{out['prediccion'].iloc[0]}**")
        probs = out[[c for c in out.columns if c.startswith("prob_")]].T
        probs.columns = ["probabilidad"]
        st.bar_chart(probs)
        st.dataframe(probs)

# ----------------------------------------------------------------------------
# 3. Lotes
# ----------------------------------------------------------------------------
with tab_batch:
    st.subheader("Predicción por lotes")
    st.write("Suba un CSV con las columnas del modelo activo (la columna objetivo es opcional):")
    st.code(", ".join(artifact["features"]))
    template = pd.DataFrame([{f: artifact["reference"]["input"].get(f) for f in artifact["features"]}])
    st.download_button("Descargar plantilla CSV", template.to_csv(index=False).encode(), "plantilla.csv")
    up_b = st.file_uploader("Explorar archivos", type=["csv"], key="batch_csv")
    if up_b is not None:
        df_b = read_uploaded(up_b.getvalue())
        out, missing, extra = ml_core.predict_df(artifact, df_b)
        if missing:
            st.warning(f"Columnas faltantes (se imputan): {missing}")
        if extra:
            st.info(f"Columnas ignoradas: {extra}")
        st.dataframe(out)
        st.write("Distribución de predicciones:")
        st.bar_chart(out["prediccion"].astype(str).value_counts())
        if artifact["target"] in df_b.columns:
            acc = (out["prediccion"].astype(str) == df_b[artifact["target"]].astype(str)).mean()
            st.metric("Accuracy contra la columna objetivo incluida", f"{acc:.3f}")
        st.download_button("Descargar predicciones", out.to_csv(index=False).encode(), "predicciones.csv")

In [ ]:
%%writefile /content/ml_streamlit_project/.streamlit/config.toml
[server]
headless = true
maxUploadSize = 200

[browser]
gatherUsageStats = false

## 15. Probar Streamlit en Google Colab

In [ ]:
# Equivalente a: !streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &
import subprocess, time
subprocess.run(["pkill", "-f", "streamlit run"])
log = open("/content/streamlit.log", "w")
st_proc = subprocess.Popen(
    ["streamlit", "run", "/content/ml_streamlit_project/app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
    cwd="/content/ml_streamlit_project", stdout=log, stderr=subprocess.STDOUT,
)
time.sleep(10)
print(open("/content/streamlit.log").read())
!curl -s -o /dev/null -w "Health check HTTP %{http_code}\n" http://localhost:8501/_stcore/health

Túnel temporal para verlo desde el navegador (solo para pruebas, **no** es el despliegue final).
La contraseña que pide localtunnel es la IP pública de Colab que imprime la primera línea.
Si la red bloquea el túnel, basta con que el log anterior muestre que Streamlit inició.
Detenga la celda (■) para cerrar el túnel.

In [ ]:
!curl -s https://loca.lt/mytunnelpassword; echo
!npx --yes localtunnel --port 8501

## 16. Pruebas mínimas antes de Docker

In [ ]:
import numpy as np

loaded = joblib.load(PROJECT / "model.joblib")

# 1) Paridad: el artefacto recargado reproduce la predicción de referencia
diff = ml_core.check_parity(loaded)
assert diff <= 1e-9, diff
print("Paridad verificada, diferencia máxima:", diff)

# 2) Las columnas de entrada coinciden con las de entrenamiento
assert loaded["features"] == [c for c in df.columns if c not in drop_columns + [TARGET]]

# 3) Tres casos distintos + valores extremos + categoría inválida + columna faltante/sobrante
casos = df.drop(columns=[TARGET]).sample(3, random_state=1).copy()
extremo = casos.iloc[[0]].copy()
for c in loaded["numeric_features"]:
    extremo[c] = loaded["numeric_stats"][c]["max"] * 10
if loaded["categorical_features"]:
    extremo[loaded["categorical_features"][0]] = "categoria_nunca_vista"
lote = pd.concat([casos, extremo]).drop(columns=[loaded["features"][0]])
lote["columna_extra"] = 1
lote.iloc[1, 0] = np.nan
out, missing, extra = ml_core.predict_df(loaded, lote)
print("Faltantes:", missing, "| Ignoradas:", extra)
display(out.iloc[:, -(len(loaded["classes"]) + 1):])

# 4) Entrenar con otro dataset del repo (lo mismo que hace la pestaña 1 de la app)
other = ml_core.read_csv_auto("https://raw.githubusercontent.com/wilmerpineda/machine_learning_ii_course_public/"
                              "main/datasets/public/Default_Clientes.csv")
art2 = ml_core.train(other, "Default", ml_core.suggest_drop_columns(other, "Default"), "logistic_regression")
print("Default_Clientes ->", {k: round(v, 4) for k, v in art2["metrics"].items()})

# 5) Ejemplo de entrada/salida para el README
ejemplo = pd.DataFrame([loaded["reference"]["input"]])
EJEMPLO_OUT, _, _ = ml_core.predict_df(loaded, ejemplo)
display(EJEMPLO_OUT)

## 17. Archivos para Docker
`requirements.txt` se escribe con **las versiones exactas instaladas en este Colab**: el `model.joblib`
debe cargarse en Docker con la misma versión de scikit-learn con la que se entrenó.

In [ ]:
from importlib.metadata import version
pkgs = ["streamlit", "pandas", "numpy", "scikit-learn", "joblib", "openpyxl"]
reqs = "\n".join(f"{p}=={version(p)}" for p in pkgs) + "\n"
(PROJECT / "requirements.txt").write_text(reqs)
print(reqs)
print("Python de Colab:", sys.version.split()[0])

In [ ]:
%%writefile /content/ml_streamlit_project/.dockerignore
.git
.venv
__pycache__
*.pyc
.ipynb_checkpoints
.env
*.log

In [ ]:
# Dockerfile del PDF §17. La versión de Python de la imagen base se iguala a la de Colab
# (PDF: python:3.12-slim) para que model.joblib se cargue con las mismas librerías.
PY = f"{sys.version_info.major}.{sys.version_info.minor}"
dockerfile = f'''FROM python:{PY}-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501

CMD ["streamlit", "run", "app.py", \\
     "--server.address=0.0.0.0", \\
     "--server.port=8501"]
'''
(PROJECT / "Dockerfile").write_text(dockerfile)
print(dockerfile)

## 18. ¿Qué significa cada instrucción Docker?
| Instrucción | Significado |
|---|---|
| `FROM python:3.X-slim` | Imagen base pequeña con Python (misma versión que el Colab donde se entrenó; hoy 3.12) |
| `WORKDIR /app` | Directorio de trabajo dentro de la imagen |
| `COPY requirements.txt .` | Copia primero las dependencias para aprovechar la caché de capas |
| `RUN pip install ...` | Instala las librerías dentro de la imagen |
| `COPY . .` | Copia el proyecto (app.py, ml_core.py, model.joblib, data/, .streamlit/) |
| `EXPOSE 8501` | Documenta el puerto de Streamlit |
| `CMD ...` | Proceso que se ejecuta al iniciar el contenedor |

El Dockerfile se escribe en Colab, pero la imagen **se construye en la VM** de GCP.

## 20. README completo

In [ ]:
m = loaded["metrics"]
metricas_md = "\n".join(f"| {k} | {v:.4f} |" for k, v in m.items())
import json
ej_in = json.dumps(loaded["reference"]["input"], ensure_ascii=False, indent=2, default=str)
probs = {c: float(EJEMPLO_OUT[c].iloc[0]) for c in EJEMPLO_OUT.columns if c.startswith("prob_")}
ej_out = f"Clase predicha: {EJEMPLO_OUT['prediccion'].iloc[0]} | Probabilidades: {probs}"

readme = f"""# Reto ML supervisado: predicción de abandono (churn) con Streamlit + Docker + GCP

**Autores:** _Nombre 1_ · _Nombre 2_

## Objetivo
Predecir si un cliente de telecomunicaciones abandonará la compañía (`{TARGET}` = 1) y servir el modelo en una
aplicación Streamlit empaquetada con Docker y desplegada en una VM de Google Cloud. La app también permite
**entrenar un modelo nuevo con cualquier CSV de clasificación** subido desde el navegador.

## Dataset y unidad de observación
- Fuente: `Customer_Churn_Iran.csv`, del repositorio del curso
  (https://github.com/wilmerpineda/machine_learning_ii_course_public/tree/main/datasets/public), copiado en `data/datos.csv`.
- Cada fila es **un cliente** con su uso agregado de 12 meses. {df.shape[0]} filas y {df.shape[1]} columnas, sin valores nulos.
- Si `data/datos.csv` no existe, `train.py` lo descarga desde la URL pública.

## Ruta elegida
**Ruta A – Aprendizaje supervisado**, clasificación binaria.

## Variables utilizadas
- Numéricas: {", ".join(loaded["numeric_features"])}
- Categóricas: {", ".join(loaded["categorical_features"])}
- Excluidas: {", ".join(loaded["dropped_columns"])} (`Customer Value` se deriva del uso: riesgo de fuga de información).
- El tipo de cada variable se infiere automáticamente: texto, o enteros con 10 o menos valores distintos, cuentan como categóricas.

## Preparación de datos
Todo vive dentro de un `Pipeline` de scikit-learn (`ml_core.build_pipeline`):
- Numéricas: `SimpleImputer(median)` → `StandardScaler`.
- Categóricas: `SimpleImputer(most_frequent)` → texto → `OneHotEncoder(handle_unknown="ignore")`.
- Nombres de columna normalizados (espacios dobles). Partición 80/20 estratificada, `random_state=42`.

## Modelo y justificación
Se compararon regresión logística (modelo base del PDF) y Random Forest (300 árboles, `min_samples_leaf=5`,
`class_weight="balanced"`, configuración del notebook de semana 14). Modelo elegido: **{ml_core.MODELS[loaded["model_name"]]}**,
por su mayor recall de la clase churn y su mayor F1 ponderado.

## Métricas (conjunto de prueba, {loaded["n_rows_test"]} clientes)
| Métrica | Valor |
|---|---|
{metricas_md}

Las clases están desbalanceadas, así que se prioriza recall y F1 sobre accuracy: no detectar a un cliente que se va es el error más costoso.

## Interpretación de la salida
La app devuelve la **clase predicha** (0 = permanece, 1 = abandona) y la **probabilidad de cada clase**.
Una probabilidad alta de 1 indica que conviene una acción de retención. Es una demostración académica: no se debe
usar para decisiones reales sin validación externa.

## Estructura
```
data/datos.csv  ml_core.py  train.py  model.joblib  app.py
requirements.txt  Dockerfile  .dockerignore  .streamlit/config.toml  README.md
```

## Cómo ejecutar train.py
```bash
pip install -r requirements.txt
python train.py                                   # dataset por defecto -> model.joblib
python train.py --data otro.csv --target clase    # cualquier CSV de clasificación
```

## Cómo ejecutar Streamlit
```bash
streamlit run app.py --server.address=0.0.0.0 --server.port=8501
```
Pestañas: **1) Entrenar con nuevo CSV**, **2) Predecir un caso**, **3) Predecir por lotes (CSV)**.

## Cómo construir la imagen Docker
```bash
docker build -t ml-streamlit-app .
docker images
```

## Cómo ejecutar el contenedor
```bash
docker run --rm -p 8501:8501 ml-streamlit-app          # prueba interactiva (Ctrl+C para salir)
docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
docker ps
docker logs ml-streamlit
```
Abrir `http://IP_EXTERNA_DE_LA_VM:8501`.

## Despliegue en Google Cloud (Compute Engine)
1. Consola de GCP → Compute Engine (habilitar la API si la consola lo pide) → Crear instancia: **Ubuntu 24.04 LTS**, tamaño pequeño (por ejemplo e2-small), IP externa, etiqueta de red `streamlit-server`.
2. Botón **SSH** y luego:
```bash
sudo apt update && sudo apt install -y docker.io git
sudo systemctl enable --now docker
git clone <URL_DEL_REPOSITORIO> && cd <NOMBRE_DEL_REPOSITORIO>
ls -la                       # Dockerfile, app.py, ml_core.py, model.joblib, requirements.txt
sudo docker build -t ml-streamlit-app .
sudo docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
```
3. Firewall (Cloud Shell o VPC → Firewall):
```bash
gcloud compute firewall-rules create allow-streamlit-8501 --allow=tcp:8501 --direction=INGRESS --target-tags=streamlit-server
```
4. Abrir `http://IP_EXTERNA:8501`.
5. **Al terminar, detener o eliminar la VM** y la regla de firewall para no generar costos.

## Actualizar la aplicación desplegada
```bash
git pull
sudo docker stop ml-streamlit && sudo docker rm ml-streamlit
sudo docker build -t ml-streamlit-app .
sudo docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
```

## Diagnóstico
| Situación | Revisión |
|---|---|
| train.py falla | ruta del dataset, nombre de la columna objetivo, dependencias |
| Contenedor se detiene | `docker ps -a`, `docker logs ml-streamlit` |
| Puerto no responde | `-p 8501:8501` y `--server.address=0.0.0.0` |
| Funciona en la VM pero no desde fuera | regla de firewall, etiqueta `streamlit-server`, IP externa |
| Error al cargar model.joblib | versiones de `requirements.txt` iguales a las del entrenamiento |

## Ejemplo de entrada y salida
Entrada:
```json
{ej_in}
```
Salida: `{ej_out}`

## Reproducción completa desde Colab
Abrir `ML_Supervisado_Streamlit_Docker_GCP.ipynb` en Google Colab → *Ejecutar todo*. Se regenera todo el proyecto.
"""
(PROJECT / "README.md").write_text(readme, encoding="utf-8")
print(readme[:1500])

## 21. Empaquetar y descargar el proyecto (para subirlo a GitHub)

In [ ]:
!cd /content && rm -f ml_streamlit_project.zip && zip -qr ml_streamlit_project.zip ml_streamlit_project -x "*/__pycache__/*"
!ls -la /content/ml_streamlit_project /content/ml_streamlit_project.zip
try:
    from google.colab import files
    files.download("/content/ml_streamlit_project.zip")
except ImportError:
    print("Fuera de Colab: el zip está en /content/ml_streamlit_project.zip")

## 22–31. Despliegue en GCP y prueba cruzada (pasos manuales)
1. Crear un repositorio en GitHub y subir el **contenido descomprimido** del zip, incluido este notebook. No subir credenciales.
2. Seguir la sección *Despliegue en Google Cloud* del `README.md` (VM Ubuntu 24.04 → Docker → `git clone` → `docker build` → `docker run` → firewall → `http://IP:8501`).
3. **Prueba cruzada (PDF §31):** otra pareja recibe solo la URL del repositorio, sigue únicamente el README, construye, ejecuta, prueba un caso y registra los problemas que encuentre.
4. Detener o eliminar la VM al finalizar.

## 34. Checklist final
- [ ] Entendemos qué representa una fila · [ ] Ruta supervisada definida · [ ] Variables justificadas
- [ ] Preprocesamiento dentro del pipeline · [ ] `train.py` corre de principio a fin · [ ] `model.joblib` se genera
- [ ] `app.py` no reentrena al arrancar · [ ] Probamos Streamlit en Colab · [ ] requirements/Dockerfile/.dockerignore
- [ ] Repositorio completo · [ ] VM con SSH · [ ] Docker instalado · [ ] `docker build` OK · [ ] `docker run` activo
- [ ] `docker logs` sin errores · [ ] Firewall · [ ] App abre con la IP externa · [ ] Prueba cruzada · [ ] VM detenida/eliminada